# Phase 1 — validating short implicit-solvent MD against SAXS and NMR

Reference system: **hen egg white lysozyme**, UniProt `P00698`.

This notebook runs the full loop end to end:

1. survey the candidate SASBDB entries and **check they describe the same molecule**
2. load one experimental scattering curve, normalised to 1/A
3. source atomic coordinates through UniProt
4. prepare and simulate in implicit solvent
5. compute an ensemble SAXS profile and fit it
6. compare NMR chemical shifts to the simulated secondary structure

Lysozyme is the regression target because it is exhaustively characterised.
If the harness cannot reproduce lysozyme, nothing else it reports is
trustworthy.

> **Use a kernel from the `saxsnmr` environment.**
>
> On Windows, conda keeps its compiled libraries in `<env>\Libraryin`, which only
> joins the search path when the environment is activated. A Jupyter kernel launches
> `python.exe` directly, so it normally misses them, and the first call into NumPy's
> linear algebra then kills the kernel with no exception and no traceback — reported
> only as *"the kernel crashed"*.
>
> Importing `saxsnmr` repairs PATH itself, so the notebook works whichever way the
> kernel was started. That is why the cell below imports it before anything else.

In [ ]:
# Environment check. Run this first.
#
# Importing saxsnmr before anything else matters: on Windows it repairs PATH
# so the conda environment's compiled libraries can be found. Without that,
# the first call into NumPy's linear algebra kills the kernel outright -- no
# exception, no traceback, just "the kernel crashed".
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import saxsnmr

print("python:", sys.executable)
if saxsnmr.DLL_PATH_ADDED:
    print("repaired PATH with", len(saxsnmr.DLL_PATH_ADDED), "library directories")
else:
    print("PATH already complete (environment was activated)")

checks = {
    "numpy linalg":   "import numpy as np; np.linalg.lstsq(np.eye(3), np.ones(3), rcond=None)",
    "scipy optimize": "from scipy.optimize import least_squares; least_squares(lambda p: p - 3.0, [0.0])",
    "mdtraj":         "import mdtraj",
    "openmm import":  "import openmm",
    "openmm runtime": ("import openmm as mm; s = mm.System(); s.addParticle(1.0); "
                       "mm.Context(s, mm.VerletIntegrator(0.001), "
                       "mm.Platform.getPlatformByName('CPU'))"),
    "pdbfixer":       "import pdbfixer",
}

failed = []
for name, snippet in checks.items():
    try:
        exec(snippet, {})
        print(f"  OK    {name}")
    except Exception as exc:
        print(f"  FAIL  {name}: {type(exc).__name__}: {exc}")
        failed.append(name)

if failed:
    print()
    print("Unusable:", ", ".join(failed))
    print("Select the 'Python (saxsnmr)' kernel, listed under Jupyter Kernels.")
    print("If the kernel dies instead of printing this, it was started without")
    print("the environment's library directory on PATH -- see the README.")
else:
    import openmm
    print("\nenvironment OK -- openmm", openmm.__version__)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))

from saxsnmr import (Fetcher, prepare, run_md, benchmark, ensemble_fit,
                     frame_tables, radius_of_gyration, summarise_constructs,
                     load_trajectory)
from saxsnmr.saxs import ScatteringTables, fit
from saxsnmr.nmr import secondary_shifts, csi_assignment, best_offset

WORK = Path("work")
WORK.mkdir(exist_ok=True)
fetcher = Fetcher(cache_dir=WORK / "cache")
UNIPROT = "P00698"

## 1. Which entries actually describe the same molecule?

A shared UniProt accession identifies a *protein*, not a *sample*. Entries
differ in construct, tags, truncation and oligomeric state.

Read this table before trusting any join. Look for disagreement in
`length`, and for `guinier_rg` / `pddf_dmax` that do not match the rest —
those entries are a different physical system. Note also that `angular_unit`
varies between entries; the loader normalises it, but never assume.

In [ ]:
codes = fetcher.search_sasbdb("lysozyme")
print(f"{len(codes)} candidate entries")

survey = summarise_constructs(fetcher, codes[:12])
survey[["code", "uniprot", "length", "guinier_rg", "pddf_dmax", "angular_unit", "type_of_curve"]]

### Reading the survey

Monomeric lysozyme is 129 residues with Rg ~1.4 nm and Dmax ~4.2 nm. Entries
reporting a much larger Rg or Dmax are aggregates or fibrils rather than the
folded monomer, and must not be averaged together with it.

In [ ]:
CODE = "SASDMJ2"   # monomeric lysozyme

curve = fetcher.sasbdb_curve(CODE)
print(curve)
print(f"Guinier Rg = {curve.guinier_rg} nm, Dmax = {curve.pddf_dmax} nm")

# Restrict to where an implicit hydration shell is valid (see README).
curve = curve.trim(q_max=0.50)
print("after trim:", curve)

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(curve.q, curve.intensity, yerr=curve.sigma, fmt=".", ms=3,
            lw=0.5, alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("q  (1/A)")
ax.set_ylabel("I(q)  (arb.)")
ax.set_title(f"{CODE} experimental")
plt.tight_layout()

## 2. Coordinates

SASBDB links a structure for only about 15% of entries, so coordinates come
from PDBe/SIFTS keyed on UniProt, falling back to AlphaFold.

Preparation strips pre-existing hydrogens, resolves alternate conformations and
selects a single chain. Lysozyme's best structure, `2vb1` at 0.65 A, needs all
three: it resolves its own hydrogens, carries altloc codes A/B/C, and labels two
different chains `A`.

In [ ]:
for hit in fetcher.best_structures(UNIPROT)[:5]:
    print(f"  {hit.pdb_id} chain {hit.chain_id}  {hit.resolution} A  {hit.method}")

structure, provenance = fetcher.structure_for(UNIPROT)
print("\nusing:", provenance)

prep = prepare(structure, WORK / "prepared.pdb", chain="A")
print(f"{prep.n_residues} residues, {prep.n_atoms} atoms")
print(prep.sequence)

## 3. How fast can this machine actually simulate?

Run this once. Throughput varies by more than an order of magnitude between
platforms, and every decision about trajectory length depends on the real number
rather than an estimate.

**Do not use OpenCL on integrated Intel graphics.** `python -m
openmm.testInstallation` on the development machine reported forces differing
from the CPU platform by 0.13, where CPU and Reference agree to 6e-06. A
platform that computes the wrong forces will still happily produce a
trajectory. Check your own output before trusting any platform other than CPU.

In [ ]:
rates = benchmark(prep, ps=2.0, platforms=["CPU"])   # OpenCL is wrong here; Reference is very slow
for name, ns_day in rates.items():
    print(f"  {name:<10} {ns_day:8.2f} ns/day" if np.isfinite(ns_day)
          else f"  {name:<10}      unavailable")

## 4. Short implicit-solvent MD

Implicit solvent is what puts this on a laptop. It also biases towards
over-compact structures, and Rg is exactly what SAXS measures most sensitively —
so treat the fit below as a check that the workflow is sound, not as a verdict
on the force field.

In [ ]:
# run_md's tuned defaults give about 6.5-8.5 ns/day for this system on a
# laptop CPU -- roughly eleven times the obvious settings, mostly from a
# shorter nonbonded cutoff. 5 ps therefore takes about a minute; 50 ps is
# around ten. Raise production_ps once the benchmark above tells you your rate.
result = run_md(prep, WORK, production_ps=5.0, equilibration_ps=2.0,
                frame_interval_ps=0.5, platform="CPU")
print(result)

In [ ]:
import mdtraj as mdt

# load_trajectory pairs the DCD with its topology and converts units.
traj = load_trajectory(result.trajectory, top=result.topology)
rg = radius_of_gyration(traj)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(len(rg)) * 0.5, rg, lw=1)
if curve.guinier_rg:
    ax.axhline(curve.guinier_rg * 10, color="crimson", ls="--",
               label=f"experimental Guinier Rg = {curve.guinier_rg * 10:.1f} A")
    ax.legend()
ax.set_xlabel("time (ps)")
ax.set_ylabel("Rg (A)")
ax.set_title("Compaction check")
plt.tight_layout()
print(f"simulated Rg = {rg.mean():.2f} +/- {rg.std():.2f} A")

## 5. SAXS: ensemble profile and fit

The experiment measures one ensemble-averaged curve, so the model must be
averaged over frames *before* fitting rather than fitting frames individually.

`c1` (excluded volume) and `c2` (hydration contrast) are free parameters. They
are reported alongside chi-squared deliberately — with two parameters floating,
a good fit alone does not establish much.

In [ ]:
ens = ensemble_fit(result, curve, stride=1)
print(ens)

static = load_trajectory(prep.path)
st = frame_tables(static, 0)
single = fit(st, curve.q, curve.intensity, curve.sigma)
print("single crystal structure:", single)

In [ ]:
fig, (ax, axr) = plt.subplots(2, 1, figsize=(6.5, 6), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
ax.errorbar(curve.q, curve.intensity, yerr=ens.sigma, fmt=".", ms=3, lw=0.5,
            alpha=0.5, label="experiment")
ax.plot(curve.q, single.model, lw=1.2, label=f"crystal structure, chi2={single.chi2:.2f}")
ax.plot(curve.q, ens.model, lw=1.5, label=f"MD ensemble, chi2={ens.chi2:.2f}")
ax.set_yscale("log")
ax.set_ylabel("I(q)")
ax.legend()
ax.set_title(f"{CODE} vs model")

axr.axhline(0, color="k", lw=0.5)
axr.plot(curve.q, (ens.model - curve.intensity) / ens.sigma, lw=1)
axr.set_xlabel("q  (1/A)")
axr.set_ylabel("residual / sigma")
plt.tight_layout()

## 6. NMR: chemical shifts against simulated secondary structure

Predicting raw shifts needs SPARTA+, SHIFTX2 or UCBShift, none of which builds
on Windows. The portable route is secondary chemical shifts: subtracting
random-coil values leaves a signal whose sign reports helix versus strand, which
can be compared with DSSP over the trajectory.

This checks that the ensemble has the right secondary structure where NMR says
it should. It is a weaker claim than reproducing every shift, and it is the
claim this notebook can actually support.

In [ ]:
ids = fetcher.bmrb_ids(UNIPROT)
print(f"{len(ids)} BMRB entries for {UNIPROT}: {ids[:10]}")

shifts = None
for bid in ids:
    df = fetcher.bmrb_shifts(bid)
    if df is not None and {"CA", "CB"} & set(df["atom"].str.upper()):
        n_ca = (df["atom"].str.upper() == "CA").sum()
        if n_ca > 50:
            shifts, used = df, bid
            break
print(f"using BMRB {used}: {len(shifts)} assigned shifts, {n_ca} CA")
shifts.head()

In [ ]:
sec = secondary_shifts(shifts)
csi = csi_assignment(sec)
print(csi["csi"].value_counts().to_dict())

dssp = mdt.compute_dssp(traj, simplified=True)
resids = np.array([r.resSeq for r in traj.topology.residues])

offset, comparison = best_offset(csi, dssp, resids)
print(f"\nbest numbering offset: {offset:+d}")
print(comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
t = comparison.table
colors = {"H": "crimson", "E": "steelblue", "C": "lightgrey"}
ax.bar(t["resid"], t["d_diff"], color=[colors.get(c, "grey") for c in t["csi"]])
ax.axhline(0.7, color="k", lw=0.5, ls=":")
ax.axhline(-0.7, color="k", lw=0.5, ls=":")
ax.set_xlabel("residue")
ax.set_ylabel("secondary shift  dCA - dCB (ppm)")
ax.set_title(f"NMR secondary shifts vs MD DSSP "
             f"({comparison.agreement:.0%} agreement over {comparison.n_compared} residues)")
plt.tight_layout()

## 7. What this does and does not establish

**Supported by the run above**

- the fetch-and-join layer resolves a molecule to a curve, a structure and shifts
- the SAXS calculator reproduces an experimental curve from coordinates
- the MD ensemble is consistent with the measured Rg and secondary structure

**Not supported**

- any claim about force field accuracy. Implicit solvent biases compaction, and
  `c1`/`c2` absorb part of the remaining discrepancy.
- anything in the wide-angle regime, which is out of scope for this project.
- any statement about a mutant, unless experimental data for that mutant exists.